### Conexão com data lake no DB
Desenvolver um notebook de setup que monta (mount) ou conecta ao Azure Data Lake Gen2 utilizando as chaves de acesso (Service Principal ou Access Key), garantindo que você consiga listar os arquivos da camada Raw/Bronze.

In [0]:
pip install adlfs pandas python-dotenv

In [0]:
import os
import pandas as pd
import adlfs
from dotenv import load_dotenv

# Carregar .env
caminho_env = None
for tentativa in [".env", "../.env", "../../.env"]:
    if os.path.exists(tentativa):
        caminho_env = tentativa
        break

if caminho_env:
    load_dotenv(dotenv_path=caminho_env)
    print(f"✅ Arquivo .env carregado: {caminho_env}")
else:
    raise FileNotFoundError("⚠️ Arquivo .env não encontrado.")

In [0]:
# Credenciais ADLS
tenant_id = os.getenv("ADLS_TENANT_ID")
client_id = os.getenv("ADLS_CLIENT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

storage_account = "internshipdatalake"
container_name = "raw"
raw_folder = "real-time-data"

if not tenant_id or not client_id or not client_secret:
    raise ValueError("⚠️ Credenciais do ADLS não encontradas no .env.")

os.environ["AZURE_TENANT_ID"] = tenant_id
os.environ["AZURE_CLIENT_ID"] = client_id
os.environ["AZURE_CLIENT_SECRET"] = client_secret

fs = adlfs.AzureBlobFileSystem(
    account_name=storage_account,
    tenant_id=tenant_id,
    client_id=client_id,
    client_secret=client_secret
)

print("🔍 Validando conexão com o Data Lake...")

raw_path = f"{container_name}/{raw_folder}"
bronze_path = f"{container_name}/squad2/bronze"

print("📂 Listando Raw:")
arquivos_raw = fs.ls(raw_path)
for arquivo in arquivos_raw[:20]:
    print(arquivo)

print("📂 Listando Bronze:")
try:
    arquivos_bronze = fs.ls(bronze_path)
    for arquivo in arquivos_bronze[:20]:
        print(arquivo)
except Exception as e:
    print("⚠️ Bronze ainda não encontrada ou vazia.")
    print(f"Caminho testado: {bronze_path}")
    print(f"Detalhe: {e}")

In [0]:
print("🔍 Procurando arquivo de rastreamento no Data Lake...")

# Primeiro lista os arquivos do caminho correto
todos_arquivos = fs.find(raw_path)

print("📂 Primeiros arquivos encontrados:")
for arquivo in todos_arquivos[:30]:
    print(arquivo)

caminho_real_no_azure = None
formato_detectado = None

# Agora procura arquivo de rastreamento
for arquivo in todos_arquivos:
    nome = arquivo.lower()

    if "rastreamento" in nome or "entrega" in nome or "tracking" in nome:
        caminho_real_no_azure = arquivo
        formato_detectado = arquivo.split(".")[-1].lower()
        break

if not caminho_real_no_azure:
    raise FileNotFoundError("⚠️ Nenhum arquivo de rastreamento foi encontrado no Data Lake.")

# Remove o nome do container do caminho
caminho_relativo = caminho_real_no_azure.replace(container_name + "/", "")

# Monta o caminho final ABFS
caminho_final = (
    f"abfs://{container_name}@{storage_account}.dfs.core.windows.net/"
    f"{caminho_relativo}"
)

print(f"✅ Arquivo localizado: {caminho_final}")
print(f"📦 Formato detectado: {formato_detectado.upper()}")

In [0]:
# Leitura dinâmica com Pandas
credenciais_pandas = {
    "account_name": storage_account,
    "client_id": client_id,
    "client_secret": client_secret,
    "tenant_id": tenant_id
}

print(f"📥 Lendo arquivo no formato {formato_detectado.upper()}...")

if formato_detectado == "parquet":
    df_pandas = pd.read_parquet(caminho_final, storage_options=credenciais_pandas)

elif formato_detectado == "csv":
    df_pandas = pd.read_csv(caminho_final, storage_options=credenciais_pandas)

elif formato_detectado == "json":
    df_pandas = pd.read_json(caminho_final, storage_options=credenciais_pandas)

else:
    raise TypeError(f"⚠️ Formato não suportado: {formato_detectado}")

df_rastreamento = spark.createDataFrame(df_pandas)

print("✅ Dados carregados no Spark!")
display(df_rastreamento)
df_rastreamento.printSchema()

In [0]:
# Configuração SQL Server
jdbc_host = os.getenv("SQL_HOST")
jdbc_db = os.getenv("SQL_DATABASE")
jdbc_user = os.getenv("SQL_USERNAME")
jdbc_pass = os.getenv("SQL_PASSWORD")

nome_tabela_destino = "squad2.ecommerce_rastreamento"

print(f"📤 Gravando dados na tabela: {nome_tabela_destino}")

(
    df_rastreamento.write
    .format("sqlserver")
    .option("host", jdbc_host)
    .option("port", "1433")
    .option("database", jdbc_db)
    .option("user", jdbc_user)
    .option("password", jdbc_pass)
    .option("dbtable", nome_tabela_destino)
    .option("encrypt", "true")
    .option("trustServerCertificate", "false")
    .mode("overwrite")
    .save()
)

print("🚀 Tabela squad2.ecommerce_rastreamento criada com sucesso no SQL Server!")